# BERT on SST-2

Fine-tuning `bert-base-uncased` on the SST-2 sentiment dataset from GLUE. Two epochs is enough to get a decent number. GLUE's test labels are hidden, so I evaluate on the validation split.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

set_seed(42)
MODEL = "bert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
raw = load_dataset("glue", "sst2")
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["sentence"], truncation=True, max_length=128)

data = raw.map(tokenize, batched=True, remove_columns=["sentence", "idx"])
print(data)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((logits.argmax(-1) == labels).mean())}

args = TrainingArguments(
    output_dir="outputs/bert-sst2",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    tokenizer=tok,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
print(trainer.evaluate())

## Try it on my own sentences

In [ ]:
model.eval()
texts = [
    "I loved every minute of this.",
    "A total waste of two hours.",
    "It was fine, I guess.",
    "Not bad at all.",
    "I wanted to like it, but I couldn't.",
]
enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
with torch.no_grad():
    probs = model(**enc).logits.softmax(-1)

for text, p in zip(texts, probs):
    print(f"{p[1]:.2f} positive  |  {text}")